# Rolling Test Overlay

Set `ROLLING_RAW_CSV` to a `rolling_test_raw.csv` file, then run all cells to plot the true test series with non-overlapping rolling forecast windows.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from experiment.rolling_forecast.artifacts import select_non_overlapping_windows
from experiment.rolling_forecast.metrics import compute_plot_loss, format_plot_loss

## Parameters

In [ ]:
ROLLING_RAW_CSV = Path("artifacts/Time-LLM_exog_fusion/20260420_163957/rolling_test_raw.csv")
MODEL_NAME = "Time-LLM_exog_fusion"

SAVE_FIGURE = False
OUTPUT_PATH = None

## Helpers

In [ ]:
REQUIRED_COLUMNS = [
    "origin_id",
    "origin_date",
    "horizon",
    "target_date",
    "y_true",
    "y_pred",
    "error",
]


def _as_integer_series(values: pd.Series, column_name: str) -> pd.Series:
    numeric = pd.to_numeric(values, errors="raise")
    numeric_array = numeric.to_numpy(dtype=float)
    if not np.isfinite(numeric_array).all():
        raise ValueError(f"{column_name} must contain only finite values")
    if not np.equal(numeric_array, np.floor(numeric_array)).all():
        raise ValueError(f"{column_name} must contain integer values")
    return numeric.astype(int)


def load_rolling_raw(csv_path: Path) -> pd.DataFrame:
    csv_path = Path(csv_path).expanduser()
    if not csv_path.exists():
        raise FileNotFoundError(f"rolling raw CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"rolling raw CSV is missing required columns: {missing_columns}")
    if df.empty:
        raise ValueError("rolling raw CSV is empty")

    df = df.loc[:, REQUIRED_COLUMNS].copy()
    df["origin_id"] = _as_integer_series(df["origin_id"], "origin_id")
    df["horizon"] = _as_integer_series(df["horizon"], "horizon")
    if (df["horizon"] <= 0).any():
        raise ValueError("horizon must contain positive values")

    df["origin_date"] = pd.to_datetime(df["origin_date"])
    df["target_date"] = pd.to_datetime(df["target_date"])
    for column in ["y_true", "y_pred", "error"]:
        df[column] = pd.to_numeric(df[column], errors="raise")

    return df.sort_values(["origin_date", "origin_id", "horizon", "target_date"]).reset_index(drop=True)


def build_truth_series(rolling_raw_df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = [column for column in ["target_date", "y_true"] if column not in rolling_raw_df.columns]
    if missing_columns:
        raise ValueError(f"rolling raw data is missing required columns: {missing_columns}")

    df = rolling_raw_df.loc[:, ["target_date", "y_true"]].copy()
    df["target_date"] = pd.to_datetime(df["target_date"])
    df["y_true"] = pd.to_numeric(df["y_true"], errors="raise")

    unique_true_counts = df.groupby("target_date")["y_true"].nunique(dropna=False)
    conflicting_dates = unique_true_counts[unique_true_counts > 1]
    if not conflicting_dates.empty:
        sample_dates = ", ".join(date.strftime("%Y-%m-%d") for date in conflicting_dates.index[:5])
        raise ValueError(f"found multiple y_true values for target_date: {sample_dates}")

    return (
        df.groupby("target_date", as_index=False)["y_true"]
        .first()
        .sort_values("target_date")
        .reset_index(drop=True)
    )


def plot_overlay_from_rolling_csv(
    csv_path: Path,
    model_name: str | None = None,
    save_path: Path | None = None,
) -> tuple[Figure, Axes]:
    rolling_raw_df = load_rolling_raw(csv_path)
    truth_df = build_truth_series(rolling_raw_df)
    horizon = int(rolling_raw_df["horizon"].max())

    selected_windows = select_non_overlapping_windows(
        rolling_raw_df=rolling_raw_df,
        target_dates=pd.DatetimeIndex(truth_df["target_date"]),
        window_size=horizon,
        window_step=horizon,
    )
    if len(selected_windows) != len(truth_df):
        raise ValueError(
            "selected overlay windows do not match the true test series length: "
            f"{len(selected_windows)} vs {len(truth_df)}"
        )

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.plot(
        truth_df["target_date"],
        truth_df["y_true"],
        color="black",
        linewidth=3,
        label="True Load (Test Set)",
    )

    group_labels = selected_windows["group_label"].drop_duplicates().tolist()
    colors = plt.cm.tab20(np.linspace(0, 1, len(group_labels)))
    for color, group_label in zip(colors, group_labels):
        group_df = (
            selected_windows.loc[selected_windows["group_label"] == group_label]
            .sort_values("target_date")
            .reset_index(drop=True)
        )
        group_loss = compute_plot_loss(
            y_true=group_df["y_true"],
            y_pred=group_df["y_pred"],
            loss_name="MAPE",
        )
        group_plot_label = f"{group_label} ({format_plot_loss('MAPE', group_loss)})"
        ax.plot(
            group_df["target_date"],
            group_df["y_pred"],
            color=color,
            linewidth=2,
            marker="o",
            markersize=4,
            label=group_plot_label,
        )

    title = "Rolling Test Prediction Overlay"
    if model_name:
        title = f"{title} ({model_name})"
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily Electricity Load")
    ax.grid(True, alpha=0.25)
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=min(4, max(1, len(group_labels))),
        frameon=False,
    )
    fig.subplots_adjust(bottom=0.28)

    if save_path is not None:
        save_path = Path(save_path).expanduser()
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()
    return fig, ax

## Coverage Check

In [ ]:
rolling_raw_df = load_rolling_raw(ROLLING_RAW_CSV)
truth_df = build_truth_series(rolling_raw_df)
horizon = int(rolling_raw_df["horizon"].max())
selected_windows = select_non_overlapping_windows(
    rolling_raw_df=rolling_raw_df,
    target_dates=pd.DatetimeIndex(truth_df["target_date"]),
    window_size=horizon,
    window_step=horizon,
)

print(f"horizon: {horizon}")
print(f"selected overlay rows: {len(selected_windows)}")
print(f"unique target dates: {truth_df['target_date'].nunique()}")
print(f"selected groups: {selected_windows['group_label'].nunique()}")
print(
    "coverage ok:",
    pd.DatetimeIndex(selected_windows["target_date"]).sort_values().equals(
        pd.DatetimeIndex(truth_df["target_date"]).sort_values()
    ),
)

first_group = selected_windows["group_label"].iloc[0]
first_group_df = selected_windows.loc[selected_windows["group_label"] == first_group]
first_group_loss = compute_plot_loss(first_group_df["y_true"], first_group_df["y_pred"], "MAPE")
print(f"first window: {first_group} {format_plot_loss('MAPE', first_group_loss)}")

## Plot

In [ ]:
save_path = None
if SAVE_FIGURE:
    save_path = Path(OUTPUT_PATH) if OUTPUT_PATH is not None else Path("rolling_test_overlay_from_csv.png")

fig, ax = plot_overlay_from_rolling_csv(ROLLING_RAW_CSV, MODEL_NAME, save_path)
if save_path is not None:
    print(f"Saved overlay plot to: {save_path.resolve()}")